In [1]:
# Answer these questions:

# 1. What happens if you start consumer_filter.py AFTER the producer has finished?
#    (Hint: check auto_offset_reset)
#
# ANSWER: The consumer still reads all messages from the beginning of the topic.
# Kafka does not delete messages when they are consumed — messages stay in the topic
# for the configured retention period (7 days by default). With auto_offset_reset='earliest',
# if group_id='filter-group' has never been registered with Kafka before, the consumer
# starts reading from offset 0 (the very first message) and processes the full history.
# If this group_id was used previously, Kafka remembers the last committed offset and
# the consumer resumes from where it left off, without re-reading old messages.

# 2. What happens if two consumers have the SAME group_id?
#
# ANSWER: Kafka treats them as a single consumer group and DISTRIBUTES the topic's
# partitions across both consumers — each partition is assigned to exactly one consumer
# in the group. With the 'transactions' topic having 3 partitions, one consumer might
# get 2 partitions and the other gets 1. Each consumer only sees the messages from its
# assigned partitions, not the full stream. Together the group still processes every
# message exactly once. This is Kafka's horizontal scaling mechanism (load balancing).
#
# In contrast, two consumers with DIFFERENT group_ids each receive a complete, independent
# copy of the stream — which is the pattern used in this lab (filter-group, enrich-group,
# count-group, stats-group).

# 3. What is the difference between stateless and stateful processing?
#    Give one example of each from this lab.
#
# ANSWER:
# Stateless processing handles each message independently — no information is kept between
# messages. The output for a given message depends only on the message itself. It is simple,
# easy to scale, and immune to crashes (restart and continue).
#   Example from this lab: consumer_filter.py — for each transaction we just check
#   `if tx['amount'] > 1000`. No memory of previous transactions is needed.
#
# Stateful processing requires MEMORY between messages — the output depends on the
# accumulated history. State is kept in variables (dicts, counters) that live OUTSIDE
# the message loop. It is more powerful but more fragile: when the script stops, all
# in-memory state is lost.
#   Example from this lab: consumer_count.py — `store_counts[store] += 1` only makes
#   sense if we remember the count from previous messages. The Counter() dict persists
#   across loop iterations.
#
# This in-memory limitation is exactly why production stream processing frameworks
# (Spark Structured Streaming in Lab 3-4) persist state to disk via checkpointLocation.

In [ ]:
from kafka import KafkaConsumer
from collections import defaultdict
from datetime import datetime, timedelta
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='velocity-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

# STATE: for each user, a list of recent transaction timestamps
user_timestamps = defaultdict(list)

# Configuration
WINDOW_SECONDS = 60
THRESHOLD = 3       # alert if MORE than this many tx in the window

# Track users we've already alerted on to avoid spamming
already_alerted = set()

print(f"Velocity anomaly detector — alert if user has > {THRESHOLD} "
      f"transactions in {WINDOW_SECONDS}s\n")

msg_count = 0
alert_count = 0

for message in consumer:
    tx = message.value
    user = tx['user_id']
    # Producer writes timestamp via datetime.now().isoformat() — same format here
    tx_time = datetime.fromisoformat(tx['timestamp'])
    msg_count += 1

    # 1) Drop timestamps older than the window from this user's history
    cutoff = tx_time - timedelta(seconds=WINDOW_SECONDS)
    user_timestamps[user] = [
        t for t in user_timestamps[user] if t >= cutoff
    ]

    # 2) Add the current timestamp
    user_timestamps[user].append(tx_time)

    # 3) Check the rule
    count_in_window = len(user_timestamps[user])
    if count_in_window > THRESHOLD:
        # Only print alert if we haven't already alerted on this user
        # (or you can comment out this check to see every offending event)
        alert_key = (user, tx_time.replace(second=0, microsecond=0))
        if alert_key not in already_alerted:
            already_alerted.add(alert_key)
            alert_count += 1
            print(
                f"🚨 ALERT #{alert_count}: user {user} made "
                f"{count_in_window} transactions in the last {WINDOW_SECONDS}s "
                f"(latest tx: {tx['tx_id']} at {tx_time.strftime('%H:%M:%S')})"
            )
            # Show the actual timestamps that triggered the alert
            times_str = ", ".join(t.strftime('%H:%M:%S') for t in user_timestamps[user])
            print(f"   Window contents: [{times_str}]")
            print(f"   Tip: investigate user {user} for possible fraud or bot activity.\n")

Velocity anomaly detector — alert if user has > 3 transactions in 60s

🚨 ALERT #1: user u14 made 4 transactions in the last 60s (latest tx: TX0452 at 17:55:14)
   Window contents: [17:54:55, 17:55:03, 17:55:13, 17:55:14]
   Tip: investigate user u14 for possible fraud or bot activity.

🚨 ALERT #2: user u01 made 4 transactions in the last 60s (latest tx: TX8826 at 17:55:17)
   Window contents: [17:55:00, 17:55:07, 17:55:10, 17:55:17]
   Tip: investigate user u01 for possible fraud or bot activity.

🚨 ALERT #3: user u11 made 4 transactions in the last 60s (latest tx: TX4501 at 17:55:25)
   Window contents: [17:54:59, 17:55:20, 17:55:24, 17:55:25]
   Tip: investigate user u11 for possible fraud or bot activity.

🚨 ALERT #4: user u12 made 4 transactions in the last 60s (latest tx: TX0084 at 17:55:40)
   Window contents: [17:54:57, 17:55:01, 17:55:27, 17:55:40]
   Tip: investigate user u12 for possible fraud or bot activity.

🚨 ALERT #5: user u09 made 4 transactions in the last 60s (latest 